## First I aggregated all the T1 images and Masks into their own folders (they are currently in subject folders) but keep them in the same directory as the subject folders. 
## All Images and masks are now here:
/home/rbielski/Approx_Numeracy/Aggregated_Images_and_masks
## In sub-folders:
/home/rbielski/Approx_Numeracy/Aggregated_Images_and_masks/Images
## And
/home/rbielski/Approx_Numeracy/Aggregated_Images_and_masks/Masks



## Second, Register Images and Masks, and then normalize, and output them here:
/home/rbielski/stroke_cleaned/Approx_Numeracy_Processed/Registered_Normalized_Images
## and here:
/home/rbielski/stroke_cleaned/Approx_Numeracy_Processed/Registered_Normalized_Masks

## Using process:
Resample mask to the native T1: resample_mask_to_t1(mask, t1, mask_t1)
Register T1 -> MNI: ants_register(t1, prefix, tpl, reg_bin, use_2mm=True) (runs rigid, affine, SyN via ANTs)
Apply transforms to T1 (warp to MNI): ants_apply(t1, tpl, prefix, t1_mni, apply_bin, nn=False)
Apply transforms to mask (nearest-neighbor): ants_apply(mask_t1, tpl, prefix, mask_mni, apply_bin, nn=True) then binarize (>0.5) and save
Normalization (always after obtaining t1_mni)

Load t1_mni as array, call normalize_t1(vol):
Take nonzero voxels, clip to 1st–99th percentiles
Z-score (subtract mean / std)
Rescale to [0,1]
Save normalized volume

In [1]:
import sys, shutil
from pathlib import Path

PROJ = Path("/home/rbielski/stroke_cleaned/Approx_Numeracy_Processed")
sys.path.insert(0, str(PROJ))

from data_prep.prep_utils import DatasetConfig, run_prep

# Input directories
IMAGES_DIR = Path("/home/rbielski/Approx_Numeracy/Aggregated_Images_and_masks/Images")
MASKS_DIR  = Path("/home/rbielski/Approx_Numeracy/Aggregated_Images_and_masks/Masks")

# Intermediate output (nested slug folders, xfm, etc.)
PREP_OUT = PROJ / "prep_intermediate"

# Final flat output directories
REG_IMAGES = PROJ / "Registered_Normalized_Images"
REG_MASKS  = PROJ / "Registered_Normalized_Masks"
REG_IMAGES.mkdir(parents=True, exist_ok=True)
REG_MASKS.mkdir(parents=True, exist_ok=True)

print("Images dir exists:", IMAGES_DIR.exists(), f"({len(list(IMAGES_DIR.glob('*.nii.gz')))} files)")
print("Masks dir exists: ", MASKS_DIR.exists(),  f"({len(list(MASKS_DIR.glob('*.nii.gz')))} files)")


Images dir exists: True (104 files)
Masks dir exists:  True (104 files)


In [2]:
ds = DatasetConfig(
    name="ApproxNumeracy",
    images_dir=IMAGES_DIR,
    masks_dir=MASKS_DIR,
    t1_glob="*.nii.gz",
    mask_glob="*.nii.gz",
    already_mni=False,
    overwrite=False,
)

outputs = run_prep([ds], out_root=PREP_OUT)
print("\nPrep complete. Dataset roots:", outputs)


[ApproxNumeracy] image root: /home/rbielski/Approx_Numeracy/Aggregated_Images_and_masks/Images | mask root: /home/rbielski/Approx_Numeracy/Aggregated_Images_and_masks/Masks
[ApproxNumeracy] globs: t1=*.nii.gz masks=*.nii.gz
[ApproxNumeracy] images: 104 masks: 104 pairs found: 104


100%|██████████| 3.47M/3.47M [00:00<00:00, 5.41MB/s]
100%|██████████| 452k/452k [00:00<00:00, 1.93MB/s]


>> /home/rbielski/stroke_cleaned/Approx_Numeracy_Processed/tools/ants/bin/antsRegistration -d 3 -r [/home/rbielski/stroke_cleaned/Approx_Numeracy_Processed/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Approx_Numeracy/Aggregated_Images_and_masks/Images/sub-085_T1w.nii.gz,1] -m Mattes[/home/rbielski/stroke_cleaned/Approx_Numeracy_Processed/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Approx_Numeracy/Aggregated_Images_and_masks/Images/sub-085_T1w.nii.gz,1,32,Regular,0.25] -t Rigid[0.1] -c 1000x500x250 -s 3x2x1vox -f 4x2x1 -m Mattes[/home/rbielski/stroke_cleaned/Approx_Numeracy_Processed/data/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Approx_Numeracy/Aggregated_Images_and_masks/Images/sub-085_T1w.nii.gz,1,32,Regular,0.25] -t Affine[0.1] -c 1000x500x250 -s 3x2x1vox -f 4x2x1 -m CC[/home/rbielski/stroke_

In [3]:
# Collect normalized T1s and cleaned masks into flat final output directories
for ds_root in outputs:
    t1_norm_dir = ds_root / "mni_1mm_ants_fixed" / "t1_norm"
    masks_clean_dir = ds_root / "mni_1mm_ants_fixed" / "masks_clean"

    t1s = sorted(t1_norm_dir.glob("*.nii.gz"))
    masks = sorted(masks_clean_dir.glob("*.nii.gz"))

    for f in t1s:
        shutil.copy2(f, REG_IMAGES / f.name)
    for f in masks:
        shutil.copy2(f, REG_MASKS / f.name)

    print(f"Copied {len(t1s)} normalized T1s  → {REG_IMAGES}")
    print(f"Copied {len(masks)} cleaned masks  → {REG_MASKS}")

print("\nDone!")


Copied 104 normalized T1s  → /home/rbielski/stroke_cleaned/Approx_Numeracy_Processed/Registered_Normalized_Images
Copied 104 cleaned masks  → /home/rbielski/stroke_cleaned/Approx_Numeracy_Processed/Registered_Normalized_Masks

Done!


## View the normalized T1s and masks in MNI space to check quality.

##idk why it outputs 3 times


In [1]:
from data_prep.viewer import show_viewer_dirs

show_viewer_dirs(
    images_dir="/home/rbielski/stroke_cleaned/Approx_Numeracy_Processed/Registered_Normalized_Images",
    masks_dir="/home/rbielski/stroke_cleaned/Approx_Numeracy_Processed/Registered_Normalized_Masks",
)


Output()